# 02 — Condition-stratified differential expression

**Goal:** a log2FC signature for every (perturbation, condition) pair.

## The design constraint

Every contrast is perturbation vs. **condition-matched control**. Never pool
controls across conditions. IFN-γ stimulation and TIL co-culture shift the
baseline transcriptome enormously, and pooling would let that shift leak into
every perturbation's signature — producing a screen in which everything is a
hit and none of it means anything.

## The replication caveat

There are no biological replicates here. Pseudo-replicates are random splits
of one sample: they give the negative-binomial model a within-group variance
term, but systematically understate biological variance, so p-values are
anti-conservative. Treat the DE ranking as a *ranking*. The permutation-tested
E-distance in nb03 is the honest effect-size check.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

from src.config import load_config, load_panels, paths, set_seed
from src.plotting import apply_style, condition_palette, savefig

cfg = load_config()
panels = load_panels()
P = paths(cfg)
SEED = set_seed(cfg)
apply_style(cfg)

sc.settings.verbosity = 1
print(f"repo: {P.root}")
print(f"seed: {SEED}")


In [ ]:
import mudata as md
mdata = md.read(P.data_interim / "frangieh_qc.h5mu")
rna, adt = mdata["rna"], mdata["adt"]
s = cfg["schema"]["obs"]
rna

## 1. HVG selection — on control cells only

Selecting variable genes on the full matrix selects partly on the effect being
measured. Same leakage failure mode as running feature selection before the
cross-validation split — which is exactly the bug worth avoiding twice.

In [ ]:
ctrl_mask = rna.obs[s["perturbation"]].astype(str) == cfg["schema"]["control_label"]
print(f"control cells: {ctrl_mask.sum():,}")

ctrl = rna[ctrl_mask].copy()
sc.pp.normalize_total(ctrl, target_sum=1e4)
sc.pp.log1p(ctrl)
sc.pp.highly_variable_genes(ctrl, n_top_genes=cfg["de"]["n_hvg"],
                            batch_key=s["condition"])
hvg = ctrl.var_names[ctrl.var["highly_variable"]].tolist()
print(f"HVGs selected on controls: {len(hvg)}")

## 2. Pseudobulk

In [ ]:
from src.pseudobulk import assign_pseudoreplicates, make_pseudobulk

assign_pseudoreplicates(rna, cfg)
counts, meta = make_pseudobulk(rna, cfg)
print(counts.shape)
meta.head()

## 3. DE per contrast

For each condition, run every perturbation against that condition's own
controls.

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

ctrl_label = cfg["schema"]["control_label"]
de_results = {}

for cond in meta["condition"].unique():
    sub = meta[meta["condition"] == cond]
    perts = [p for p in sub["perturbation"].unique() if p != ctrl_label]
    print(f"\n=== {cond}: {len(perts)} perturbations ===")

    for pert in perts:
        keep = sub[sub["perturbation"].isin([pert, ctrl_label])].index
        if sub.loc[keep, "perturbation"].nunique() < 2:
            continue
        # TODO: guard against too-few reps per group before fitting
        dds = DeseqDataSet(
            counts=counts.loc[keep, hvg],
            metadata=sub.loc[keep, ["perturbation"]],
            design="~perturbation",
            refit_cooks=True, quiet=True,
        )
        dds.deseq2()
        st = DeseqStats(dds, contrast=["perturbation", pert, ctrl_label], quiet=True)
        st.summary()
        if cfg["de"]["lfc_shrink"]:
            st.lfc_shrink(coeff=f"perturbation_{pert}_vs_{ctrl_label}")
        de_results[(pert, cond)] = st.results_df

print(f"\ntotal contrasts: {len(de_results)}")

## 4. Sanity checks

Both must pass before proceeding. If either fails, something upstream broke
and the rest of the project is built on sand.

### 4a. Self-knockdown

Each perturbation's own target transcript should go down in its own arm
(frameshift-induced nonsense-mediated decay). Perturbations that fail this are
candidate editing failures — knowing which ones is worth a figure, and worth a
sentence in the limitations section.

In [ ]:
rows = []
for (pert, cond), df in de_results.items():
    if pert in df.index:
        rows.append({"perturbation": pert, "condition": cond,
                     "self_lfc": df.loc[pert, "log2FoldChange"],
                     "self_padj": df.loc[pert, "padj"]})
self_kd = pd.DataFrame(rows)

fig, ax = plt.subplots()
for cond, g in self_kd.groupby("condition"):
    ax.hist(g["self_lfc"].dropna(), bins=40, alpha=0.55, label=cond,
            color=condition_palette(cfg).get(cond))
ax.axvline(0, c="k", lw=1)
ax.set_xlabel("log2FC of target gene in its own perturbation")
ax.legend()
savefig(fig, "02_self_knockdown", cfg)

print(f"median self-LFC: {self_kd['self_lfc'].median():.2f}")
print("suspect (LFC > 0):")
self_kd[self_kd["self_lfc"] > 0].sort_values("self_lfc", ascending=False).head(15)

### 4b. IFN-γ response in unperturbed cells

Control cells under IFN-γ must show JAK/STAT and antigen-presentation induction. This is the strongest available positive control for the whole pipeline.

In [ ]:
# TODO: contrast control cells IFNg vs Control, then check that
# panels["gene_sets"]["ifng_jak_stat"] and ["antigen_presentation_mhc_i"]
# come up induced.
print("[stub] IFNg positive control")

## 5. Signature matrix

In [ ]:
from src.stats import signature_matrix

sig = signature_matrix(de_results, value_col="log2FoldChange", genes=hvg)
sig.to_parquet(P.data_processed / "signatures.parquet")
print(sig.shape)
sig.iloc[:5, :5]